# linspace-out-param — ex2: contrast linspace(out=) with .copy_(linspace(...)) — same data, different alloc

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `linspace-out-param`. Running the final beacon cell reports progress against the `PyTorch: linspace out= param` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: linspace out= param` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`linspace-out-param`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "linspace-out-param"
DD_SUBTOPIC = "PyTorch: linspace out= param"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `linspace(out=)` vs `.copy_(linspace(...))` — same data, different alloc

Ex1 used `linspace(out=buf)` to fill a pre-allocated buffer. The deepening move contrasts that against the obvious alternative — `buf.copy_(t.linspace(...))` — and shows they produce the SAME numeric result but DIFFERENT allocation behaviour:

```python
# Option A: out= fills buf directly, no temp.
t.linspace(0., 1., 100, out=buf)

# Option B: linspace returns a new tensor, then copy_ overwrites buf.
# Thread dtype=buf.dtype so the temp doesn't get cast on copy.
buf.copy_(t.linspace(0., 1., 100, dtype=buf.dtype))
```

**Both preserve `buf.data_ptr()`.** `out=` writes in place; `copy_` also writes in place into the existing storage. So downstream pointers to `buf` stay valid in either case.

**Option A allocates 0 extra tensors. Option B allocates 1 (the linspace return value), then discards it.** In a tight loop, the temp is GC-able but you pay allocator churn. `out=` is the zero-temp idiom.

### Exercise 2 — contrast linspace(out=) with .copy_(linspace(...)) — same data, different alloc

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze whether `linspace(out=buf)` and `buf.copy_(linspace(...))` produce identical numeric results and identical `data_ptr()` for `buf`, demonstrating both are in-place writes that preserve the buffer identity.
> Keywords: linspace, out, copy_, data_ptr, allocation
> ```

**KCs targeted:** `out-vs-copy-equivalence`, `data-ptr-preserved-in-place`

Implement `ex2_fill_two_ways(buf_a, buf_b, start, end)`. Both buffers have the same shape — a 1-D float tensor. You will fill EACH using a different idiom and return a dict reporting whether (a) the resulting data is equal, (b) each buffer's `data_ptr()` was preserved.

Steps:

1. Record the original `data_ptr()` of each buffer: `ptr_a_before = buf_a.data_ptr()`, `ptr_b_before = buf_b.data_ptr()`.
2. Fill `buf_a` using `t.linspace(start, end, buf_a.numel(), out=buf_a)` — the `out=` idiom.
3. Fill `buf_b` using `buf_b.copy_(t.linspace(start, end, buf_b.numel(), dtype=buf_b.dtype))` — the temp-then-copy idiom. Threading `dtype=buf_b.dtype` makes the temp match the buffer so the copy is bit-exact (no float32→float64 round-trip).
4. Return a dict:
   - `'equal'`: True iff `t.equal(buf_a, buf_b)`.
   - `'ptr_a_preserved'`: True iff `buf_a.data_ptr() == ptr_a_before`.
   - `'ptr_b_preserved'`: True iff `buf_b.data_ptr() == ptr_b_before`.

Both buffers MUST be mutated in place (caller will inspect them afterward).

In [ ]:
def ex2_fill_two_ways(buf_a, buf_b, start, end):
    ptr_a_before = buf_a.data_ptr()
    ptr_b_before = buf_b.data_ptr()
    # Idiom A: out= writes directly into buf_a, no temp.
    t.linspace(start, end, buf_a.numel(), out=buf_a)
    # Idiom B: linspace returns a fresh tensor (dtype-matched), copy_ overwrites buf_b in place.
    buf_b.copy_(t.linspace(start, end, buf_b.numel(), dtype=buf_b.dtype))
    return {
        'equal': bool(t.equal(buf_a, buf_b)),
        'ptr_a_preserved': buf_a.data_ptr() == ptr_a_before,
        'ptr_b_preserved': buf_b.data_ptr() == ptr_b_before,
    }


<details><summary>Solution</summary>

```python
def ex2_fill_two_ways(buf_a, buf_b, start, end):
    ptr_a_before = buf_a.data_ptr()
    ptr_b_before = buf_b.data_ptr()
    # Idiom A: out= writes directly into buf_a, no temp.
    t.linspace(start, end, buf_a.numel(), out=buf_a)
    # Idiom B: linspace returns a fresh tensor (dtype-matched), copy_ overwrites buf_b in place.
    buf_b.copy_(t.linspace(start, end, buf_b.numel(), dtype=buf_b.dtype))
    return {
        'equal': bool(t.equal(buf_a, buf_b)),
        'ptr_a_preserved': buf_a.data_ptr() == ptr_a_before,
        'ptr_b_preserved': buf_b.data_ptr() == ptr_b_before,
    }
```

**Both idioms preserve `data_ptr()`.** `out=` writes into the existing storage. `.copy_()` does the same — it overwrites the destination's storage with the source's values, NOT replacing the storage. So any caller that captured `buf.data_ptr()` before the fill still has a valid pointer afterward.

**The difference is allocator pressure.** `out=` allocates zero extra tensors. `.copy_(linspace(...))` allocates ONE fresh tensor (the linspace return value), copies it into `buf_b`, then discards it. In a tight loop you pay allocator + GC churn.

**`t.equal` over `t.allclose` for this assertion.** `t.equal` is exact-element equality. Since both idioms ultimately call the SAME linspace kernel against the same dtype, the bits are identical — exact equality is the strongest statement we can make.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()